In [ ]:
import subprocess, os

script = """
set -e  # Abortar si cualquier comando falla

# ── 1. Clonar el repo ──────────────────────────────────────────────
cd /content
if [ ! -d cactus ]; then
  git clone --depth=1 https://github.com/cactus-compute/cactus
fi

# ── 2. Descargar NDK r27c ──────────────────────────────────────────
NDK_ZIP="/content/ndk.zip"
NDK_DIR="/content/android-ndk-r27c"

if [ ! -d "$NDK_DIR" ]; then
  echo "[1/7] Descargando NDK r27c..."
  curl -L -o "$NDK_ZIP" \
    "https://dl.google.com/android/repository/android-ndk-r27c-linux.zip"
  echo "[2/7] Descomprimiendo..."
  unzip -q "$NDK_ZIP" -d /content
fi

# ── 3. Permisos ────────────────────────────────────────────────────
echo "[3/7] Aplicando chmod 755..."
chmod -R 755 "$NDK_DIR"

# ── 4. Exportar variables EN ESTE MISMO PROCESO ───────────────────
export ANDROID_NDK_HOME="$NDK_DIR"
export PATH="$NDK_DIR/toolchains/llvm/prebuilt/linux-x86_64/bin:$PATH"
echo "[4/7] PATH actualizado."

# ── 5. Verificar que clang++ existe ───────────────────────────────
echo "[5/7] Verificando clang++..."
which clang++ || { echo "ERROR: clang++ no encontrado en PATH"; exit 1; }
clang++ --version

# ── 6. Compilar ────────────────────────────────────────────────────
echo "[6/7] Compilando libcactus.so..."
cd /content/cactus/android
bash build.sh

# ── 7. Verificar output ───────────────────────────────────────────
echo "[7/7] Verificando output..."
ls -lh /content/cactus/android/libcactus.so || { echo "ERROR: libcactus.so no generado"; exit 1; }
echo "✅ Compilación exitosa."
"""

result = subprocess.run(
    script,
    shell=True,
    executable="/bin/bash",   # forzar bash (no sh) para que 'export' funcione bien
    capture_output=False,     # mostrar stdout/stderr en tiempo real
    text=True,
)

if result.returncode != 0:
    print(f"\n❌ Falló con código {result.returncode}")
else:
    print("\n✅ Todo OK.")


✅ Todo OK.


In [ ]:
from google.colab import files

files.download("/content/cactus/android/libcactus.so")
files.download("/content/cactus/android/Cactus.kt")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>